# 三因子打分 · Generative Agent 风格

**本 Notebook 目标：** 演示记忆检索时如何融合 **相关性、近期性、重要性** 三个信号，对候选记忆做加权排序（Generative Agent 论文思路）。

> 与 `02_long_term_memory` 中的综合得分同源；本课拆开三因子，展示 min-max 归一化与权重融合细节。

## 三因子原理

| 因子 | 含义 | 来源 |
|------|------|------|
| **相关性 Relevance** | 与当前查询的语义相似度 | 向量检索 raw score |
| **近期性 Recency** | 越久未访问得分越低 | `exp(-decay_per_hour × hours)` |
| **重要性 Importance** | 写入时标注的长期权重 | 0~1，无需再归一化 |

融合前对 **相关性、近期性** 做 min-max 归一化（量纲对齐）；重要性已在 0~1。

$$\text{final} = w_{rel} \cdot rel_n + w_{rec} \cdot rec_n + w_{imp} \cdot imp_n$$

## 打分流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear'}}}%%
flowchart TD
    IN["向量检索得分 + 候选记忆"] --> REL["相关性：min-max 归一化 → rel_n"]
    IN --> REC["近期性：exp(-衰减 × 小时) → min-max 归一化 → rec_n"]
    IN --> IMP["重要性：直接取 imp（已 0~1）"]
    REL --> SUM["加权融合<br/>w_rel×rel_n + w_rec×rec_n + w_imp×imp"]
    REC --> SUM
    IMP --> SUM
    SUM --> RANK["按综合分降序 → 取 top-k"]
```

### 流程中的参数说明

| 参数 | 类型 | 含义 | 代码中的来源 |
|------|------|------|-------------|
| `rel_scores` | `List[float]` | 向量检索原始相似度，未归一化 | 模拟检索返回，如 `[0.82, 0.55, …]` |
| `rel_n` | `List[float]` | 相关性归一化到 0~1 | `norm_minmax(rel_scores)` |
| `hours` | `float` | 距上次访问过了多少小时 | `(now - last_access_ts) / 3600` |
| `decay_per_hour` | `float` | 近期性衰减系数，默认 `0.2` | 越大忘得越快；1 小时前 ≈ `exp(-0.2×1)` |
| `rec_raw` | `List[float]` | 近期性原始分 | `exp(-decay_per_hour × hours)` |
| `rec_n` | `List[float]` | 近期性归一化到 0~1 | `norm_minmax(rec_raw)` |
| `imp` | `float` | 重要性，写入时标注 | `Mem.importance`，已在 0~1 |
| `w_rel` | `float` | 相关性权重，默认 `0.5` | 调高 → 更偏向「和 query 像」 |
| `w_rec` | `float` | 近期性权重，默认 `0.3` | 调高 → 更偏向「最近访问过」 |
| `w_imp` | `float` | 重要性权重，默认 `0.2` | 调高 → 更偏向「标注为重要的记忆」 |
| `final` | `float` | 单条记忆综合分 | `w_rel×rel_n + w_rec×rec_n + w_imp×imp` |
| `top-k` | `int` | 最终返回条数 | 按 `final` 降序取前 k 条 |

> **输入侧**：每条候选记忆还需 `Mem.last_access_ts`（上次访问时间戳）和 `Mem.text`（内容，供展示）。  
> **归一化**：`rel_n`、`rec_n` 在同一批候选内做 min-max，最高分 → 1，最低分 → 0；`imp` 不再归一化。

---
# Part 1 · 核心实现

纯 Python 标准库，无第三方依赖。

In [1]:
# Part 1 · 依赖与编码（纯标准库，无第三方包）
import math   # exp：计算近期性衰减
import sys
import time   # time.time()：与 last_access_ts 算距现在的小时数
from dataclasses import dataclass
from typing import List, Tuple

try:
    sys.stdout.reconfigure(encoding="utf-8")  # 终端下正确输出中文
except Exception:
    pass  # Jupyter OutStream 不支持 reconfigure，跳过即可


@dataclass
class Mem:
    """单条候选记忆：向量检索命中后，再经三因子重排。"""
    id: str
    text: str
    importance: float       # 重要性，写入时标注，范围 0~1
    last_access_ts: float   # Unix 时间戳，用于算「距现在多久未访问」


def norm_minmax(values: List[float]) -> List[float]:
    """Min-max 归一化到 [0, 1]；全相等时返回 1.0。"""
    if not values:
        return []
    lo, hi = min(values), max(values)
    if hi - lo < 1e-9:
        return [1.0 for _ in values]  # 候选分值相同，不做区分
    return [(v - lo) / (hi - lo) for v in values]


def generative_agent_style_scores(
    rel_scores: List[float],   # 向量检索原始相似度（量纲因模型而异）
    memories: List[Mem],
    w_rel: float = 0.5,        # 相关性权重
    w_rec: float = 0.3,        # 近期性权重
    w_imp: float = 0.2,        # 重要性权重
    decay_per_hour: float = 0.2,  # 近期性衰减系数，越大忘得越快
) -> List[float]:
    """相关性 + 指数近期性 + 重要性的加权融合，返回每条记忆的综合分。"""
    now = time.time()
    recency_raw = []
    for m in memories:
        hours = max(0.0, (now - m.last_access_ts) / 3600.0)
        recency_raw.append(math.exp(-decay_per_hour * hours))  # 越久未访问 → 越接近 0

    rel_n = norm_minmax(rel_scores)   # 相关性归一化，与近期性量纲对齐
    rec_n = norm_minmax(recency_raw)
    imp_n = [m.importance for m in memories]  # 已在 0~1，无需再归一化

    out = []
    for i in range(len(memories)):
        out.append(w_rel * rel_n[i] + w_rec * rec_n[i] + w_imp * imp_n[i])
    return out


def rank_memories(
    rel_scores: List[float],
    memories: List[Mem],
    **kwargs,  # 透传 w_rel / w_rec / w_imp / decay_per_hour
) -> List[Tuple[float, Mem, dict]]:
    """返回 (综合分, 记忆, 因子明细) 列表，按综合分降序。"""
    decay = kwargs.get("decay_per_hour", 0.2)
    now = time.time()
    recency_raw = [
        math.exp(-decay * max(0.0, (now - m.last_access_ts) / 3600.0))
        for m in memories
    ]
    rel_n = norm_minmax(rel_scores)
    rec_n = norm_minmax(recency_raw)
    imp_n = [m.importance for m in memories]
    final = generative_agent_style_scores(rel_scores, memories, **kwargs)

    ranked = []
    for i, m in enumerate(memories):
        detail = {
            "rel_raw": rel_scores[i],   # 归一化前
            "rel_n": rel_n[i],
            "rec_raw": recency_raw[i],
            "rec_n": rec_n[i],
            "imp": imp_n[i],
            "final": final[i],
        }
        ranked.append((final[i], m, detail))
    ranked.sort(key=lambda x: x[0], reverse=True)
    return ranked


def explain_ranking(
    rel_scores: List[float],
    memories: List[Mem],
    **kwargs,
) -> None:
    """打印排序结果及三因子拆解，方便课堂讲解。"""
    w_rel = kwargs.get("w_rel", 0.5)
    w_rec = kwargs.get("w_rec", 0.3)
    w_imp = kwargs.get("w_imp", 0.2)
    decay = kwargs.get("decay_per_hour", 0.2)
    now = time.time()

    print(f"权重：rel={w_rel}, rec={w_rec}, imp={w_imp} | decay_per_hour={decay}\n")
    print(f"{'#':<3} {'final':>7} {'rel_n':>7} {'rec_n':>7} {'imp':>5}  {'hours':>6}  text")
    print("-" * 80)

    for i, (score, mem, d) in enumerate(rank_memories(rel_scores, memories, **kwargs), 1):
        hours_ago = (now - mem.last_access_ts) / 3600.0
        print(
            f"{i:<3} {score:>7.4f} {d['rel_n']:>7.3f} {d['rec_n']:>7.3f} {d['imp']:>5.2f}  "
            f"{hours_ago:>6.1f}h  {mem.text}"
        )
        print(
            f"    raw: rel={d['rel_raw']:.3f}  rec={d['rec_raw']:.3f}  "
            f"→ norm 后加权求和"
        )


_NOTEBOOK_READY = True
print("[Part 1] Mem / norm_minmax / rank_memories 已定义")

[Part 1] Mem / norm_minmax / rank_memories 已定义


---
# Part 2 · 演示：4 条记忆排序

模拟向量检索返回 4 条候选记忆，观察三因子如何改变最终排名（**相关性高但很久未访问** vs **近期但相关性略低**）。

In [ ]:
if not globals().get("_NOTEBOOK_READY", False):
    raise RuntimeError("请先运行 Part 1")

now = time.time()

memories = [
    Mem("m1", "用户偏好：沟通风格简洁。", 0.9, now - 3600),          # 1 小时前
    Mem("m2", "用户住在上海，偏好早上开会。", 0.6, now - 86400),     # 1 天前
    Mem("m3", "上周讨论过 RAG 重排方案。", 0.4, now - 7 * 86400),   # 1 周前
    Mem("m4", "项目使用向量库存 FAQ。", 0.7, now - 1800),           # 30 分钟前
]

# 模拟向量检索原始相似度（未归一化，量纲各异）
rel_scores = [0.82, 0.55, 0.48, 0.71]

print("=== 三因子打分（Generative Agent 风格）===\n")
explain_ranking(rel_scores, memories)

ranked = rank_memories(rel_scores, memories)

=== 三因子打分（Generative Agent 风格）===

权重：rel=0.5, rec=0.3, imp=0.2 | decay_per_hour=0.2

#     final   rel_n   rec_n   imp   hours  text
--------------------------------------------------------------------------------
1    0.9515   1.000   0.905  0.90     1.0h  用户偏好：沟通风格简洁。
    raw: rel=0.820  rec=0.819  → norm 后加权求和
2    0.7782   0.676   1.000  0.70     0.5h  项目使用向量库存 FAQ。
    raw: rel=0.710  rec=0.905  → norm 后加权求和
3    0.2257   0.206   0.009  0.60    24.0h  用户住在上海，偏好早上开会。
    raw: rel=0.550  rec=0.008  → norm 后加权求和
4    0.0800   0.000   0.000  0.40   168.0h  上周讨论过 RAG 重排方案。
    raw: rel=0.480  rec=0.000  → norm 后加权求和

[Part 2] 演示通过 ✓


# Part 3 · 调权重实验

提高 **近期性权重** `w_rec=0.5`，观察「30 分钟前访问的 m4」**综合分是否明显上升、与 m1 差距是否缩小**——讲解互动：「如果产品更强调新鲜度，该调哪个权重？」

In [3]:
print("=== 实验：提高近期性权重 w_rec=0.5, w_rel=0.3 ===\n")
explain_ranking(rel_scores, memories, w_rel=0.3, w_rec=0.5, w_imp=0.2)

ranked_default = rank_memories(rel_scores, memories)
ranked_rec = rank_memories(rel_scores, memories, w_rel=0.3, w_rec=0.5, w_imp=0.2)

m4_default = next(s for s, m, _ in ranked_default if m.id == "m4")
m4_rec = next(s for s, m, _ in ranked_rec if m.id == "m4")
print(f"\nm4 综合分：{m4_default:.4f} → {m4_rec:.4f}（近期性权重提高后上升）")
print(f"Top-1 仍为 [{ranked_rec[0][1].id}]（m1 相关性+重要性双高）")
print("\n[check] 权重实验完成 ✓")

=== 实验：提高近期性权重 w_rec=0.5, w_rel=0.3 ===

权重：rel=0.3, rec=0.5, imp=0.2 | decay_per_hour=0.2

#     final   rel_n   rec_n   imp   hours  text
--------------------------------------------------------------------------------
1    0.9324   1.000   0.905  0.90     1.0h  用户偏好：沟通风格简洁。
    raw: rel=0.820  rec=0.819  → norm 后加权求和
2    0.8429   0.676   1.000  0.70     0.5h  项目使用向量库存 FAQ。
    raw: rel=0.710  rec=0.905  → norm 后加权求和
3    0.1863   0.206   0.009  0.60    24.0h  用户住在上海，偏好早上开会。
    raw: rel=0.550  rec=0.008  → norm 后加权求和
4    0.0800   0.000   0.000  0.40   168.0h  上周讨论过 RAG 重排方案。
    raw: rel=0.480  rec=0.000  → norm 后加权求和

m4 综合分：0.7782 → 0.8429（近期性权重提高后上升）
Top-1 仍为 [m1]（m1 相关性+重要性双高）

[check] 权重实验完成 ✓
